In [15]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.stats as stats, scipy.io as sio
from scipy.ndimage import gaussian_filter1d
from neo.io import BlackrockIO
from nilearn import datasets, plotting

In [16]:
patient = 18

### make new dir with only figs of clusters for easier QC


In [17]:
# make new dir with only figs of clusters for QC
figs_clust = f'../../results/2025{patient}/osort_mat/figs_clust'
os.makedirs(figs_clust, exist_ok=True)

# existing dir
figs = f'../../results/2025{patient}/osort_mat/figs'
for fig_name in glob.glob(f'{figs}/*/*.png'):

    if 'CL' in os.path.basename(fig_name) and not 'ALL' in os.path.basename(fig_name):
        dest = os.path.join(figs_clust, os.path.basename(fig_name))
        if os.path.exists(dest):
            print(f'File already exists, skipping: {dest}')
        else:
            os.system(f'cp {fig_name} {figs_clust}/') # copy to new dir

File already exists, skipping: ../../results/202518/osort_mat/figs_clust/A211_CL_2793_THM_1.png
File already exists, skipping: ../../results/202518/osort_mat/figs_clust/A200_CL_810_THM_1.png
File already exists, skipping: ../../results/202518/osort_mat/figs_clust/A214_CL_1289_THM_1.png
File already exists, skipping: ../../results/202518/osort_mat/figs_clust/A203_CL_464_THM_1.png
File already exists, skipping: ../../results/202518/osort_mat/figs_clust/A207_CL_1827_THM_1.png
File already exists, skipping: ../../results/202518/osort_mat/figs_clust/A207_CL_1988_THM_1.png
File already exists, skipping: ../../results/202518/osort_mat/figs_clust/A198_CL_244_THM_1.png
File already exists, skipping: ../../results/202518/osort_mat/figs_clust/A213_CL_2153_THM_1.png
File already exists, skipping: ../../results/202518/osort_mat/figs_clust/A202_CL_892_THM_1.png
File already exists, skipping: ../../results/202518/osort_mat/figs_clust/A223_CL_2_THM_1.png
File already exists, skipping: ../../results/20

### create chanID file, which will be used for QC

In [18]:
df_rows = []
# to get channel & unit IDs
fig_files = glob.glob(f'../../results/2025{patient}/osort_mat/figs/5/*')

for fig_file in sorted(fig_files):

    # skip raw fig files, and all-unit fig files
    if 'CL' not in fig_file or 'ALL' in fig_file: continue
    
    chanID = int(os.path.basename(fig_file).split('_')[0][1:])
    unitID = int(os.path.basename(fig_file).split('_')[2])
    df_rows.append((chanID, unitID))

# sort properly by channel then unit
df_rows_sorted = sorted(df_rows, key=lambda x: (x[0], x[1]))
    
chanID_df = pd.DataFrame(df_rows_sorted, columns=['chanID', 'unitID'])
# create new dir if it doesn't exist
os.makedirs(f'../../results/2025{patient}/records', exist_ok=True)
csv_path = f'../../results/2025{patient}/records/chanID_pt{patient}.csv'
if os.path.exists(csv_path):
    print(f'File already exists, skipping: {csv_path}')
else:
    chanID_df.to_csv(csv_path, index=False)
chanID_df

File already exists, skipping: ../../results/202518/records/chanID_pt18.csv


,chanID,unitID
0,193,421
1,193,467
2,194,179
3,196,308
4,197,203
...,...,...
65,221,3
66,222,4
67,223,2
68,223,3


### after performing manual QC, load:

In [19]:
QC_df = pd.read_csv(f'../../results/2025{patient}/records/QC_pt{patient}.csv')

# clean
QC_df['keep'] = QC_df['keep'].fillna(0)
dropped_clustIDs = QC_df[QC_df['keep'] != 1]['unitID'].astype(int).tolist()
dropped_clustIDs.extend([0, 1, 99999999])
QC_df = QC_df[~QC_df['unitID'].isin(dropped_clustIDs)].copy().reset_index(drop=True)

QC_df

,chanID,unitID,keep,notes
0,193,421,1.0,FR <.5
1,193,467,1.0,NaN
2,200,833,1.0,autocorr
3,202,871,1.0,NaN
4,202,892,1.0,FR <.5
5,203,464,1.0,"FR<.5, autocorr"
6,204,701,1.0,NaN
7,204,708,1.0,NaN
8,205,590,1.0,NaN
9,206,612,1.0,NaN


In [20]:
def getclustID2spikes(clustIDs, spikes):
    ''' return dict with keys=unique clusters, and vals = list of corresponding spikes '''
    
    dropped_clustIDs.extend([0, 1, 99999999])  # also drop these
    # initialize clust2spikes with keys as unique cluster IDs and empty list as values
    clust2spikes = {}
    for clustID, spike in zip(clustIDs, spikes):

        if clustID in dropped_clustIDs: continue

        if clustID not in clust2spikes: clust2spikes[clustID] = [] # initialize

        clust2spikes[clustID].append(spike)

    return clust2spikes

In [21]:
samp_rate = 1000000
neur_spikes_df = []

# go through OSort mat files
for mat_file in glob.glob(f'../../results/2025{patient}/osort_mat/sorted_mats/5/*_sorted_new.mat'):

    chan_mat = sio.loadmat(mat_file)

    # if chan_mat['assignedNegative'].size == 0: continue

    chanID = int(os.path.basename(mat_file).split('_')[0][1:])  # A198_sorted_new.mat -> 198
    clustIDs = chan_mat['assignedNegative'][0] # len = total n_spikes
    spikes = chan_mat['newTimestampsNegative'][0] # len = total n_spikes
    # clust_spikes_df = pd.DataFrame({'clustID': clustIDs, 'spike': spikes})

    # create dict:  clustID => [spikes], retaining only QCed units
    clust2spikes = getclustID2spikes(clustIDs, spikes) # len = # unique QCed clustIDs

    # 1 row per QCed clustID
    clust_df = pd.DataFrame([
        {
            "chanID": chanID,
            "clustID": clustID,
            "spikes": np.array(spikes)/samp_rate, #[spike/samp_rate for spike in spikes],
            "#spikes": len(spikes),
            "avgFR": len(spikes) / ((spikes[-1] - spikes[0]) / samp_rate),
        }
        for clustID, spikes in clust2spikes.items()
    ])
    neur_spikes_df.append(clust_df)

neur_spikes_df = pd.concat(neur_spikes_df, ignore_index=True)
# sort by chanID and then clustID
neur_spikes_df = neur_spikes_df.sort_values(by=['chanID', 'clustID']).reset_index(drop=True)
neur_spikes_df


,chanID,clustID,spikes,#spikes,avgFR
0,193,421,"[2.9386333333333337, 3.025166666666667, 6.5640...",572,0.346617
1,193,467,"[3.260966666666667, 5.094733333333334, 11.0821...",1250,0.757284
2,200,833,"[1.5603333333333336, 3.720666666666667, 5.7518...",1301,0.788853
3,202,871,"[3.734466666666667, 4.718, 8.183800000000002, ...",1114,0.681801
4,202,892,"[21.639133333333337, 25.3345, 26.2455000000000...",591,0.361751
5,203,464,"[4.283266666666667, 6.975800000000001, 10.0429...",684,0.414428
6,204,701,"[0.5282, 1.0729333333333335, 1.3612, 2.3284666...",1727,1.043810
7,204,708,"[5.711866666666667, 12.2751, 13.62493333333333...",1026,0.622157
8,205,590,"[3.493566666666667, 4.5576, 7.251833333333334,...",912,0.552273
9,206,612,"[0.017533333333333335, 0.6868666666666667, 3.7...",951,0.576399


In [22]:
# print lens of clustIDs in both dfs
print(f'len QC_df clustIDs: {len(QC_df["unitID"].unique())}')
print(f'len neur_spikes_df clustIDs: {len(neur_spikes_df["clustID"].unique())}')

# instead, for every channel, print clustIDs that are in QC_df but not in neur_spikes_df
for chanID in QC_df['chanID'].unique():
    qc_clustIDs = set(QC_df[QC_df['chanID'] == chanID]['unitID'])
    neur_clustIDs = set(neur_spikes_df[neur_spikes_df['chanID'] == chanID]['clustID'])
    missing_clustIDs = qc_clustIDs - neur_clustIDs
    if missing_clustIDs:
        print(f'chanID {chanID} missing clustIDs: {missing_clustIDs}')

len QC_df clustIDs: 23
len neur_spikes_df clustIDs: 23


### channel -> region -> index -> coordinates (-> atlas regions?)

In [23]:
# channels -> regions
channelInfo = sio.loadmat(
    glob.glob(f'../../results/2025{patient}/records/*ChannelMap*.mat')[0]
    )

if patient in [12, 21]: channelMap = channelInfo['ChannelMap1'].flatten()
elif patient == 18: channelMap = channelInfo['ChannelMap2'].flatten()

labelMap = channelInfo['LabelMap'].flatten() # contains region labels
labelMap = np.array([str(label.squeeze()) for label in labelMap])  # convert to str

# dict after removing nan keys
nan_mask = ~np.isnan(channelMap)
channel2label = dict(zip(channelMap[nan_mask], labelMap[nan_mask]))

neur_spikes_df['region'] = neur_spikes_df['chanID'].map(channel2label).fillna(neur_spikes_df['chanID']).apply(lambda x: str(x))
print(neur_spikes_df['region'].unique())


<ArrowStringArray>
[ 'mROFC1',  'mROFC8',  'mRACC2',  'mRACC3',  'mRACC4',  'mRACC5',  'mRACC6',
  'mRACC7', 'mRAHIP2', 'mRAHIP3', 'mRAHIP5', 'mRAHIP8']
Length: 12, dtype: str


In [ ]:
# regions -> coords

def clean_entry(x):
    while isinstance(x, (np.ndarray, list)):
        x = x[0]
    if isinstance(x, (bytes, bytearray)):
        x = x.decode("utf-8", errors="ignore")
    return str(x)

# load
electrodeInfo = sio.loadmat(f'../../results/2025{patient}/records/2025{patient}_DI_Electrodes.mat')
ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']) # region -> ID
ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?

atlas_index = 0 # NMM

# clean
region_s = ElecMapRaw[0].apply(clean_entry)                    # Series of regions
atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions

# build small tables
region2id_df = pd.DataFrame({"region": region_s.values,
                             "ID": np.arange(len(region_s))})
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),
#                                  "atlas_region": atlas_s.values})

# merge
final_neur_df = (neur_spikes_df
                .merge(region2id_df, on='region', how='left')
                .merge(id2xyz_df, on='ID', how='left')
                # .merge(xyz2atlasRegions, on='ID', how='left')
                )
final_neur_df = final_neur_df.drop(columns=['ID'])
# final_neur_df.to_csv(f'../../results/2025{patient}/records/df_spikes.csv', index=False)
final_neur_df['spikes'] = final_neur_df['spikes'].apply(lambda x: np.array(x))  # ensure arrays
parquet_path = f'../../results/2025{patient}/records/processed_data/df_spikes.parquet'
if os.path.exists(parquet_path):
    print(f'File already exists, skipping: {parquet_path}')
else:
    final_neur_df.to_parquet(parquet_path, index=False)

File already exists, skipping: ../../results/202518/records/df_spikes.parquet


### inspect

In [25]:
eg_spikes = final_neur_df['spikes'].iloc[0]
print(f'last 5 spikes (s): {eg_spikes[-5:]}')
print(f'last 5 spikes (min): {eg_spikes[-5:]/60}')
n_neurs = len(final_neur_df)
final_neur_df

last 5 spikes (s): [1641.31823333 1644.7786     1645.8907     1651.39533333 1653.17633333]
last 5 spikes (min): [27.35530389 27.41297667 27.43151167 27.52325556 27.55293889]


,chanID,clustID,spikes,#spikes,avgFR,region,x,y,z
0,193,421,"[2.9386333333333337, 3.025166666666667, 6.5640...",572,0.346617,mROFC1,5.400004,42.999992,-10.400004
1,193,467,"[3.260966666666667, 5.094733333333334, 11.0821...",1250,0.757284,mROFC1,5.400004,42.999992,-10.400004
2,200,833,"[1.5603333333333336, 3.720666666666667, 5.7518...",1301,0.788853,mROFC8,3.000004,44.199992,-10.400004
3,202,871,"[3.734466666666667, 4.718, 8.183800000000002, ...",1114,0.681801,mRACC2,6.600004,29.799992,23.199994
4,202,892,"[21.639133333333337, 25.3345, 26.2455000000000...",591,0.361751,mRACC2,6.600004,29.799992,23.199994
5,203,464,"[4.283266666666667, 6.975800000000001, 10.0429...",684,0.414428,mRACC3,6.600004,27.399993,23.199994
6,204,701,"[0.5282, 1.0729333333333335, 1.3612, 2.3284666...",1727,1.043810,mRACC4,7.800004,27.399993,23.199994
7,204,708,"[5.711866666666667, 12.2751, 13.62493333333333...",1026,0.622157,mRACC4,7.800004,27.399993,23.199994
8,205,590,"[3.493566666666667, 4.5576, 7.251833333333334,...",912,0.552273,mRACC5,7.800004,29.799992,23.199994
9,206,612,"[0.017533333333333335, 0.6868666666666667, 3.7...",951,0.576399,mRACC6,9.000004,28.599993,23.199994
